In [1]:
import pandas as pd

df = pd.read_csv("CRMLSListing_merge.csv")
df.head(5)

/var/folders/d_/gll3jlg1445_2nm3xgq_xvdc0000gn/T/ipykernel_9386/3028586889.py:3: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("CRMLSListing_merge.csv")


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1,month
0,90000.0,1075010398,miriamlara03@gmail.com,NaN,NaN,Miriam,Lara,34.097939,-117.909653,1045 N Azusa 61,...,NaN,2.0,Covina Valley Unified,91722,NaN,0.0,NaN,NaN,1045 N Azusa 61,202401
1,1500000.0,1074974457,janelle@judsonre.com,NaN,NaN,Janelle,Judson,33.121241,-117.081614,NaN,...,NaN,NaN,NaN,92025,NaN,0.0,NaN,NaN,NaN,202401
2,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,False,NaN,NaN,90067,NaN,2105.0,177861.0,NaN,2220 Avenue Of The Stars 2704,202401
3,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,False,3.0,Capistrano Unified,92677,NaN,254.0,5300.0,NaN,16 Palisades,202401
4,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,NaN,2.0,NaN,91108,NaN,NaN,9404.0,NaN,1615 Waverly Road,202401


In [2]:
num_cols = df.select_dtypes(include=["number"]).columns
cat_cols = df.select_dtypes(exclude=["number"]).columns

print("Numerical columns:", len(num_cols))
print(num_cols)

Numerical columns: 37
Index(['OriginalListPrice', 'ListingKey', 'ClosePrice', 'Latitude',
       'Longitude', 'LivingArea', 'ListPrice', 'DaysOnMarket',
       'FireplacesTotal', 'AboveGradeFinishedArea', 'ListingKeyNumeric',
       'TaxAnnualAmount', 'ParkingTotal', 'LotSizeAcres', 'YearBuilt',
       'DaysOnMarket.1', 'StreetNumberNumeric', 'LivingArea.1',
       'BathroomsTotalInteger', 'BuyerAgencyCompensation', 'TaxYear',
       'BuildingAreaTotal', 'BedroomsTotal', 'Longitude.1',
       'ElementarySchoolDistrict', 'BelowGradeFinishedArea', 'Latitude.1',
       'ListPrice.1', 'CoveredSpaces', 'Stories', 'LotSizeArea',
       'MainLevelBedrooms', 'GarageSpaces', 'AssociationFee',
       'LotSizeSquareFeet', 'MiddleOrJuniorSchoolDistrict', 'month'],
      dtype='str')


In [3]:
import pandas as pd
import numpy as np
from IPython.display import display


#  Check duplicates
# =========================
total_duplicates = df.duplicated().sum()

if "ListingId" in df.columns:
    listing_duplicates = df.duplicated(subset=["ListingId"]).sum()
else:
    listing_duplicates = None

print("=== DUPLICATE CHECK ===")
print("Total duplicate rows:", total_duplicates)
print("Duplicate ListingId:", listing_duplicates)


# Define core numerical columns
# =========================
num_keep_cols = [
    "ListPrice",
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "DaysOnMarket",
    "YearBuilt",
    "Latitude",
    "Longitude"
]

# Keep only columns that actually exist in df
num_keep_cols = [col for col in num_keep_cols if col in df.columns]

# =========================
# Generate summary statistics
# =========================
num_cols = df.select_dtypes(include=np.number).columns.tolist()
summary = df[num_cols].describe().T

report = []

for col in num_cols:
    col_data = df[col].dropna()

    missing_count = df[col].isna().sum()
    missing_pct = round(df[col].isna().mean() * 100, 2)

    if col_data.empty:
        report.append([
            col, False,
            missing_count, missing_pct,
            0, 0,
            np.nan, np.nan, np.nan, np.nan,
            np.nan, np.nan,
            0, False, False,
            "Drop"
        ])
        continue

    min_val = summary.loc[col, "min"]
    max_val = summary.loc[col, "max"]
    mean_val = summary.loc[col, "mean"]
    std_val = summary.loc[col, "std"]

    negative_count = int((col_data < 0).sum())
    zero_count = int((col_data == 0).sum())

    # =========================
    #  Detect possible outlier problems
    # =========================

    # Extreme Max
    p99 = col_data.quantile(0.99)
    extreme_max = max_val > p99 * 5 if pd.notna(p99) and p99 != 0 else False

    # High Std
    high_std = std_val > abs(mean_val) * 2 if pd.notna(mean_val) and mean_val != 0 else False

    # IQR Outliers
    q1 = col_data.quantile(0.25)
    q3 = col_data.quantile(0.75)
    iqr = q3 - q1

    if iqr > 0:
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outlier_count = int(((col_data < lower) | (col_data > upper)).sum())
    else:
        outlier_count = 0

    # =========================
    #  Mark whether each column is core
    # =========================
    is_core = col in num_keep_cols

    report.append([
        col, is_core,
        missing_count, missing_pct,
        negative_count, zero_count,
        round(min_val, 2), round(max_val, 2),
        round(mean_val, 2), round(std_val, 2),
        round(q1, 2), round(q3, 2),
        outlier_count, extreme_max, high_std,
        None
    ])

# =========================
# Create report DataFrame
# =========================
report_df = pd.DataFrame(report, columns=[
    "Column", "Is Core",
    "Missing", "Missing %",
    "Negative Count", "Zero Count",
    "Min", "Max", "Mean", "Std",
    "Q1", "Q3",
    "IQR Outliers",
    "Extreme Max ⚠️",
    "High Std ⚠️",
    "Action"
])

# =========================
#  Assign an action
# =========================
def assign_action(row):
    if row["Missing %"] > 90:
        return "Drop"
    elif row["Negative Count"] > 0:
        return "Fix Negative"
    elif row["Extreme Max ⚠️"] or row["High Std ⚠️"] or row["IQR Outliers"] > 0:
        return "Cap / Clean"
    else:
        return "OK"

report_df["Action"] = report_df.apply(assign_action, axis=1)

# =========================
# Sort the results by priority
# =========================
action_order = {
    "Drop": 0,
    "Cap / Clean": 1,
    "Fix Negative": 2,
    "OK": 3
}

report_df["ActionRank"] = report_df["Action"].map(action_order)

report_df = report_df.sort_values(
    by=["ActionRank", "Missing %"],
    ascending=[True, False]
).reset_index(drop=True)

#  Display results in groups
# =========================
print("\n=== DROP ===")
display(report_df[report_df["Action"] == "Drop"])

print("\n=== KEEP (CORE) ===")
display(
    report_df[report_df["Is Core"]]
    .sort_values(by=["ActionRank", "Missing %"])
)

print("\n=== CAP / CLEAN ===")
display(report_df[report_df["Action"] == "Cap / Clean"])

print("\n=== KEEP (OK + Fix Negative) ===")
display(report_df[report_df["Action"].isin(["OK", "Fix Negative"])])

=== DUPLICATE CHECK ===
Total duplicate rows: 0
Duplicate ListingId: 151

=== DROP ===


,Column,Is Core,Missing,Missing %,Negative Count,Zero Count,Min,Max,Mean,Std,Q1,Q3,IQR Outliers,Extreme Max ⚠️,High Std ⚠️,Action,ActionRank
0,FireplacesTotal,False,853863,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
1,AboveGradeFinishedArea,False,853863,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
2,ElementarySchoolDistrict,False,853863,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
3,CoveredSpaces,False,853863,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
4,MiddleOrJuniorSchoolDistrict,False,853863,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
5,TaxYear,False,852995,99.90,0,0,2004.0,2026.0,2023.33,2.65,2023.0,2024.00,64,False,False,Drop,0
6,BelowGradeFinishedArea,False,850442,99.60,0,2774,0.0,17424.0,96.90,497.46,0.0,0.00,0,True,True,Drop,0
7,TaxAnnualAmount,False,847769,99.29,0,1335,0.0,74460000.0,30614.77,965482.03,119.0,19629.75,345,True,True,Drop,0



=== KEEP (CORE) ===


,Column,Is Core,Missing,Missing %,Negative Count,Zero Count,Min,Max,Mean,Std,Q1,Q3,IQR Outliers,Extreme Max ⚠️,High Std ⚠️,Action,ActionRank
24,ListPrice,True,2135,0.25,0,7,0.00,443650265.0,984256.51,2167855.84,85000.00,1150000.00,51329,True,True,Cap / Clean,1
21,YearBuilt,True,69465,8.14,0,0,1776.00,2028.0,1978.40,28.04,1960.00,2001.00,1567,False,False,Cap / Clean,1
20,BathroomsTotalInteger,True,69951,8.19,0,34015,0.00,2208.0,2.50,3.03,2.00,3.00,81620,True,False,Cap / Clean,1
16,BedroomsTotal,True,103078,12.07,0,7217,0.00,149.0,3.13,1.69,2.00,4.00,5486,True,False,Cap / Clean,1
14,LivingArea,True,106472,12.47,0,841,0.00,909090909.0,3159.02,1051747.27,1182.00,2236.00,40229,True,True,Cap / Clean,1
10,ClosePrice,True,621529,72.79,0,12,0.00,820000000.0,780991.41,3461987.04,5000.00,1025000.00,10116,True,True,Cap / Clean,1
33,DaysOnMarket,True,0,0.00,36,98897,-58.00,2539.0,19.11,28.84,4.00,22.00,75160,True,False,Fix Negative,2
30,Longitude,True,111245,13.03,742369,94,-159.52,392.0,-118.36,3.89,-118.52,-117.29,121930,True,False,Fix Negative,2
28,Latitude,True,111949,13.11,7,95,-117.47,737.0,34.52,1.89,33.76,34.30,168596,True,False,Fix Negative,2



=== CAP / CLEAN ===


,Column,Is Core,Missing,Missing %,Negative Count,Zero Count,Min,Max,Mean,Std,Q1,Q3,IQR Outliers,Extreme Max ⚠️,High Std ⚠️,Action,ActionRank
8,BuyerAgencyCompensation,False,730344,85.53,0,632,0.0,2.000000e+05,1.750500e+02,2053.69,2.000000e+00,2.500000e+00,16537,True,True,Cap / Clean,1
9,BuildingAreaTotal,False,716928,83.96,0,870,0.0,1.200000e+06,3.343490e+03,11349.99,1.219000e+03,3.209000e+03,11899,True,True,Cap / Clean,1
10,ClosePrice,True,621529,72.79,0,12,0.0,8.200000e+08,7.809914e+05,3461987.04,5.000000e+03,1.025000e+06,10116,True,True,Cap / Clean,1
11,MainLevelBedrooms,False,459098,53.77,0,69994,0.0,3.333000e+03,1.930000e+00,5.60,1.000000e+00,3.000000e+00,923,True,True,Cap / Clean,1
12,AssociationFee,False,276017,32.33,0,328035,0.0,9.683480e+05,2.053200e+02,2060.85,0.000000e+00,3.110000e+02,26247,True,True,Cap / Clean,1
13,GarageSpaces,False,162135,18.99,0,122380,0.0,6.000000e+02,1.760000e+00,3.94,1.000000e+00,2.000000e+00,18885,True,True,Cap / Clean,1
14,LivingArea,True,106472,12.47,0,841,0.0,9.090909e+08,3.159020e+03,1051747.27,1.182000e+03,2.236000e+03,40229,True,True,Cap / Clean,1
15,LivingArea.1,False,106472,12.47,0,841,0.0,9.090909e+08,3.159020e+03,1051747.27,1.182000e+03,2.236000e+03,40229,True,True,Cap / Clean,1
16,BedroomsTotal,True,103078,12.07,0,7217,0.0,1.490000e+02,3.130000e+00,1.69,2.000000e+00,4.000000e+00,5486,True,False,Cap / Clean,1
17,LotSizeAcres,False,81583,9.55,0,17284,0.0,3.702600e+08,5.507400e+02,421471.26,1.200000e-01,3.800000e-01,129651,True,True,Cap / Clean,1



=== KEEP (OK + Fix Negative) ===


,Column,Is Core,Missing,Missing %,Negative Count,Zero Count,Min,Max,Mean,Std,Q1,Q3,IQR Outliers,Extreme Max ⚠️,High Std ⚠️,Action,ActionRank
28,Latitude,True,111949,13.11,7,95,-117.47,737.0,34.52,1.89,33.76,34.30,168596,True,False,Fix Negative,2
29,Latitude.1,False,111949,13.11,7,95,-117.47,737.0,34.52,1.89,33.76,34.30,168596,True,False,Fix Negative,2
30,Longitude,True,111245,13.03,742369,94,-159.52,392.0,-118.36,3.89,-118.52,-117.29,121930,True,False,Fix Negative,2
31,Longitude.1,False,111245,13.03,742369,94,-159.52,392.0,-118.36,3.89,-118.52,-117.29,121930,True,False,Fix Negative,2
32,ParkingTotal,False,52355,6.13,177,111161,-143.00,2593669.0,6.43,2906.69,1.00,3.00,32172,True,True,Fix Negative,2
33,DaysOnMarket,True,0,0.00,36,98897,-58.00,2539.0,19.11,28.84,4.00,22.00,75160,True,False,Fix Negative,2
34,DaysOnMarket.1,False,0,0.00,36,98897,-58.00,2539.0,19.11,28.84,4.00,22.00,75160,True,False,Fix Negative,2
35,Stories,False,221526,25.94,0,0,1.00,2.0,1.37,0.48,1.00,2.00,0,False,False,OK,3
36,month,False,0,0.00,0,0,202401.00,202603.0,202473.10,67.40,202407.00,202508.00,0,False,False,OK,3


##Second Screening: "CAP / CLEAN" Section and "KEEP (OK + Fix Negative)" Section

Drop: BuyerAgencyCompensation, AssociationFee, StreetNumberNumeric,ListingKey,ListingKeyNumeric	

duplicate(Needs to be deleted): LivingArea.1, ListPrice.1,Latitude.1,Longitude.1,DaysOnMarket.1

keep: 
BuildingAreaTotal-Total area of the structure. 

ClosePrice	

MainLevelBedrooms

GarageSpaces

BedroomsTotal

LotSizeSquareFeet

LotSizeArea

BathroomsTotalInteger

YearBuilt

OriginalListPrice

ListPrice	


